# Merge attack and normal datasets

## Loading and cleaning

In [ ]:
# Make project helpers available from a kernel started in this folder or the repo root.
from pathlib import Path
import sys
PROJECT_ROOT = next(root for root in (Path.cwd(), *Path.cwd().parents)
                    if (root / "preprocessing").is_dir() and (root / "scenario_reconstructor").is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
normal_train=pd.read_csv("data/normal_train/normal_2_3_4.csv")
normal_test=pd.read_csv("data/normal_test/normal_5_6_7.csv")
attack_train=pd.read_csv("data/dollar_char_train/dollar_char_train_labeled.csv")
attack_test=pd.read_csv("data/dollar_char_test/dollar_char_test_labeled.csv")

In [ ]:
to_be_dropped=[col for col in normal_train.columns if "Unnamed" in col]
normal_train=normal_train.drop(to_be_dropped, axis=1)
normal_test=normal_test.drop(to_be_dropped, axis=1)

In [ ]:
normal_train.info()

In [ ]:
normal_test.info()

In [ ]:
to_be_dropped=["phase_idx", "phase_name", "phase_number"] + [col for col in attack_train.columns if "Unnamed" in col]
print(to_be_dropped)

In [ ]:
attack_train=attack_train.drop(to_be_dropped, axis=1)
attack_test=attack_test.drop(to_be_dropped, axis=1)

In [ ]:
attack_train.info()

In [ ]:
attack_test.info()

## Convert to IPv4

In [ ]:
mappings_attack={}

for _, row in attack_train.iterrows():
    src_mac=row["src_mac"]
    dst_mac=row["dst_mac"]
    src_ip=row["src_ip"]
    dst_ip=row["dst_ip"]
    if ":" not in src_ip:
        mappings_attack[src_mac]=src_ip
    if ":" not in dst_ip:
        mappings_attack[dst_mac]=dst_ip

for _, row in attack_test.iterrows():
    src_mac=row["src_mac"]
    dst_mac=row["dst_mac"]
    src_ip=row["src_ip"]
    dst_ip=row["dst_ip"]
    if ":" not in src_ip:
        mappings_attack[src_mac]=src_ip
    if ":" not in dst_ip:
        mappings_attack[dst_mac]=dst_ip        
        

In [ ]:
print(mappings_attack)

In [ ]:
mappings_normal={}

for _, row in normal_train.iterrows():
    src_mac=row["src_mac"]
    dst_mac=row["dst_mac"]
    src_ip=row["src_ip"]
    dst_ip=row["dst_ip"]
    if ":" not in src_ip:
        mappings_normal[src_mac]=src_ip
    if ":" not in dst_ip:
        mappings_normal[dst_mac]=dst_ip

for _, row in normal_test.iterrows():
    src_mac=row["src_mac"]
    dst_mac=row["dst_mac"]
    src_ip=row["src_ip"]
    dst_ip=row["dst_ip"]
    if ":" not in src_ip:
        mappings_normal[src_mac]=src_ip
    if ":" not in dst_ip:
        mappings_normal[dst_mac]=dst_ip        
        

In [ ]:
print(mappings_normal)

In [ ]:
print(mappings_attack.keys() & mappings_normal.keys())

In [ ]:
mappings = mappings_attack | mappings_normal

In [ ]:
print(mappings)

In [ ]:
normal_train["src_mac"].unique()

In [ ]:
normal_train["dst_mac"].unique()

In [ ]:
attack_train["src_mac"].unique()

In [ ]:
attack_test["src_mac"].unique()

In [ ]:
mapping_not_found_train=0
for i, row in attack_train.iterrows():
    src_mac=row["src_mac"]
    dst_mac=row["dst_mac"]
    src_ip=row["src_ip"]
    dst_ip=row["dst_ip"]
    if ":" in src_ip and src_mac in mappings:
        attack_train.loc[i, "src_ip"]=mappings[src_mac]  
    elif src_mac not in mappings:
        mapping_not_found_train+=1        
    if ":" in dst_ip and dst_mac in mappings:
        attack_train.loc[i, "dst_ip"]=mappings[dst_mac]
    elif dst_mac not in mappings:
        mapping_not_found_train+=1        

mapping_not_found_test=0
for i, row in attack_test.iterrows():
    src_mac=row["src_mac"]
    dst_mac=row["dst_mac"]
    src_ip=row["src_ip"]
    dst_ip=row["dst_ip"]
    if ":" in src_ip and src_mac in mappings:
        attack_test.loc[i, "src_ip"]=mappings[src_mac] 
    elif src_mac not in mappings:
        mapping_not_found_test+=1
    if ":" in dst_ip and dst_mac in mappings:
        attack_test.loc[i, "dst_ip"]=mappings[dst_mac]
    elif dst_mac not in mappings:
        mapping_not_found_test+=1                

In [ ]:
print(mapping_not_found_train)

In [ ]:
print(mapping_not_found_test)

In [ ]:
attack_train

In [ ]:
attack_test

In [ ]:
mapping_not_found_train=0
for i, row in normal_train.iterrows():
    src_mac=row["src_mac"]
    dst_mac=row["dst_mac"]
    src_ip=row["src_ip"]
    dst_ip=row["dst_ip"]
    if ":" in src_ip and src_mac in mappings:
        normal_train.loc[i, "src_ip"]=mappings[src_mac]    
    elif src_mac not in mappings:
        mapping_not_found_train+=1         
    if ":" in dst_ip and dst_mac in mappings:
        normal_train.loc[i, "dst_ip"]=mappings[dst_mac]
    elif dst_mac not in mappings:
        mapping_not_found_train+=1         

mapping_not_found_test=0
for i, row in normal_test.iterrows():
    src_mac=row["src_mac"]
    dst_mac=row["dst_mac"]
    src_ip=row["src_ip"]
    dst_ip=row["dst_ip"]
    if ":" in src_ip and src_mac in mappings:
        normal_test.loc[i, "src_ip"]=mappings[src_mac]    
    elif src_mac not in mappings:
        mapping_not_found_test+=1        
    if ":" in dst_ip and dst_mac in mappings:
        normal_test.loc[i, "dst_ip"]=mappings[dst_mac]
    elif dst_mac not in mappings:
        mapping_not_found_test+=1         

In [ ]:
print(mapping_not_found_train)

In [ ]:
print(mapping_not_found_test)

In [ ]:
normal_train

In [ ]:
normal_test

In [ ]:
from preprocessing.network import filter_ipv4_flows, is_ipv4
attack_test = filter_ipv4_flows(attack_test)
attack_train = filter_ipv4_flows(attack_train)

In [ ]:
normal_test = filter_ipv4_flows(normal_test)
normal_train = filter_ipv4_flows(normal_train)

In [ ]:
attack_test.info()

In [ ]:
attack_test["src_ip"].value_counts()

In [ ]:
attack_test["dst_ip"].value_counts()

In [ ]:
attack_train.info()

In [ ]:
attack_train["src_ip"].value_counts()

In [ ]:
attack_train["dst_ip"].value_counts()

In [ ]:
normal_test.info()

In [ ]:
normal_test["src_ip"].value_counts()

In [ ]:
normal_test["dst_ip"].value_counts()

In [ ]:
normal_train.info()

In [ ]:
normal_train["src_ip"].value_counts()

In [ ]:
normal_train["dst_ip"].value_counts()

In [ ]:
attack_steps = (
    attack_train
    .groupby('label', as_index=False)
    .agg(total_steps=('step_number', 'nunique'))
)

print(attack_steps)

In [ ]:
attack_steps = (
    attack_test
    .groupby('label', as_index=False)
    .agg(total_steps=('step_number', 'nunique'))
)

print(attack_steps)

## Augment normal dataset

In [ ]:
from preprocessing.augmentation import EXCLUDED_FEATURE_KEYWORDS
keywords_excluded = EXCLUDED_FEATURE_KEYWORDS

In [ ]:
from preprocessing.augmentation import augment_normal_flows

def generate_dataframe(df, start_time, repetitions):
    return augment_normal_flows(df, start_time, repetitions, excluded_keywords=keywords_excluded)

In [ ]:
dfs = [normal_train]
for i in range(1, 500):
    dfs.append(generate_dataframe(dfs[0], dfs[i-1]["bidirectional_last_seen_ms"].max(), 500))

In [ ]:
normal_train=pd.concat(dfs, ignore_index=True)
normal_train.info()

In [ ]:
normal_train

In [ ]:
dfs = [normal_test]
for i in range(1, 500):
    dfs.append(generate_dataframe(dfs[0], dfs[i-1]["bidirectional_last_seen_ms"].max(), 500))

In [ ]:
normal_test=pd.concat(dfs, ignore_index=True)
normal_test.info()

In [ ]:
normal_test

## Convert to absolute time

In [ ]:
from preprocessing.time import absolute_flow_time
convert_to_absolute_time = absolute_flow_time

In [ ]:
import pytz
from preprocessing.time import shift_capture_time
start_time = np.load("./data/dollar_char_train/dollar_char_train.npz", allow_pickle=True)
start_time = start_time['datetime']
start_time = shift_capture_time(start_time.item())

In [ ]:
normal_train['timestamp'] = normal_train['bidirectional_first_seen_ms'].apply(convert_to_absolute_time, start_time=start_time)
normal_train["label"] = "normal"
normal_train["step_number"] = -1

In [ ]:
normal_test['timestamp'] = normal_test['bidirectional_first_seen_ms'].apply(convert_to_absolute_time, start_time=start_time)
normal_test["label"] = "normal"
normal_test["step_number"] = -1

## Save augmented normal dataset

In [ ]:
normal_train.to_csv("data/normal_train.csv", index=False)
normal_test.to_csv("data/normal_test.csv", index=False)

## Concatenate the dataframes

In [ ]:
df_test=pd.concat([normal_test, attack_test], ignore_index=True)

In [ ]:
df_test.info()

In [ ]:
df_train=pd.concat([normal_train, attack_train], ignore_index=True)

In [ ]:
df_train.info()

## Save results

In [ ]:
df_train.to_csv("data/train.csv", index=False)
df_test.to_csv("data/test.csv", index=False)